In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
import click
import tkinter as tk
from tkinter import filedialog
import locan as lc


# =============================================================================
# STEP 1 — Data loading
# =============================================================================
# This lightweight wrapper is used to store localization data in a consistent
# format throughout the script. It keeps the dataframe itself in `.data` and
# stores the coordinate column names used later in the pipeline.
# =============================================================================
class LocData:
    def __init__(self, data):
        self.data = data
        self.coordinate_keys = ['position_x', 'position_y']

    @staticmethod
    def from_dataframe(df):
        return LocData(df)


# =============================================================================
# Function: load_data
# =============================================================================
# Loads localization files from disk using the appropriate LOCAN loader,
# depending on whether the files come from ELYRA or ThunderSTORM.
#
# Parameters
# ----------
# file_type : str
#     Either "ELYRA" or "THUNDERSTORM".
#
# paths : list or tuple
#     Collection of file paths selected by the user.
#
# Returns
# -------
# locdata_list : list of tuples
#     A list of (path, locdata) pairs, where:
#         - path is the original file path
#         - locdata is the loaded localization object
#
# Notes
# -----
# - The function checks that coordinate keys exist.
# - The function also ensures that the 'frame' column is present, since frame
#   filtering is used later in the workflow.
# =============================================================================
def load_data(file_type, paths):
    locdata_list = []

    for path in tqdm(paths, desc="Loading data files"):
        if file_type == "ELYRA":
            locdata = lc.load_Elyra_file(path=path)
        elif file_type == "THUNDERSTORM":
            locdata = lc.load_thunderstorm_file(path=path)
        else:
            raise ValueError("Unsupported file type. Supported types are 'ELYRA' and 'THUNDERSTORM'.")

        # Ensure that coordinate keys exist and are properly defined
        if not hasattr(locdata, 'coordinate_keys') or not locdata.coordinate_keys:
            locdata.coordinate_keys = ['position_x', 'position_y']

        # Ensure that frame information exists, since it is used for filtering
        if 'frame' not in locdata.data.columns:
            raise ValueError(f"'frame' column not found in file: {path}")

        locdata_list.append((path, locdata))

    return locdata_list


# =============================================================================
# Function: select_files_for_condition
# =============================================================================
# Opens a file dialog so the user can select all localization files belonging
# to one experimental condition (for example WT or 3NQ).
#
# Parameters
# ----------
# condition_name : str
#     Name of the condition currently being loaded.
#
# Returns
# -------
# file_type : str
#     User-entered file type ("ELYRA" or "THUNDERSTORM").
#
# file_paths : tuple
#     Paths to the selected files.
# =============================================================================
def select_files_for_condition(condition_name):
    print(f"\nSelect files for condition: {condition_name}")

    root = tk.Tk()
    root.withdraw()

    file_paths = filedialog.askopenfilenames(
        title=f"Select localization data files for {condition_name}"
    )

    file_type = input("Enter the file type (ELYRA or THUNDERSTORM): ").strip().upper()

    return file_type, file_paths


# =============================================================================
# Function: get_psf_column
# =============================================================================
# Determines which PSF-related column should be used for filtering, depending
# on file type and available columns.
#
# Parameters
# ----------
# df : pandas.DataFrame
#     Localization dataframe.
#
# file_type : str
#     Either "ELYRA" or "THUNDERSTORM".
#
# Returns
# -------
# psf_col : str
#     Name of the PSF column to use for filtering.
# =============================================================================
def get_psf_column(df, file_type):
    if file_type == "ELYRA":
        if 'psf_half_width' not in df.columns:
            raise KeyError("Expected column 'psf_half_width' not found in ELYRA data.")
        return 'psf_half_width'

    elif file_type == "THUNDERSTORM":
        if 'psf_sigma' in df.columns:
            return 'psf_sigma'
        elif 'psf_half_width' in df.columns:
            return 'psf_half_width'
        else:
            raise KeyError("Neither 'psf_sigma' nor 'psf_half_width' exist in the THUNDERSTORM data.")

    else:
        raise ValueError("Unsupported file type. Supported types are 'ELYRA' and 'THUNDERSTORM'.")


# =============================================================================
# Function: get_filtering_thresholds
# =============================================================================
# Calculates suggested default filtering thresholds from pooled data across
# all loaded files, then asks the user to accept or modify them manually.
#
# Filters included
# ----------------
# 1. Localization precision max  -> removes poorly localized detections
# 2. PSF max                     -> removes wide / poor / overlapping fits
# 3. Intensity min               -> removes very dim detections / noise
# 4. Intensity max               -> removes unusually bright outliers
# 5. Frame min                   -> removes early frames if desired
#
# These thresholds are only suggested automatically.
# The user still manually confirms or changes them via click prompts.
#
# Parameters
# ----------
# locdata_with_types : list of tuples
#     Each tuple is expected to contain:
#         (file_type, locdata)
#
# Returns
# -------
# thresholds : dict
#     Dictionary containing the user-confirmed filtering thresholds.
# =============================================================================
def get_filtering_thresholds(locdata_with_types):
    pooled_uncertainty = []
    pooled_psf = []
    pooled_intensity = []
    pooled_frame = []

    # Pool values across all files from all conditions
    for file_type, locdata in locdata_with_types:
        df = locdata.data
        psf_col = get_psf_column(df, file_type)

        required_columns = ['uncertainty', 'intensity', 'frame', psf_col]
        for col in required_columns:
            if col not in df.columns:
                raise KeyError(f"The specified column '{col}' does not exist in the data.")

        pooled_uncertainty.append(pd.to_numeric(df['uncertainty'], errors='coerce'))
        pooled_psf.append(pd.to_numeric(df[psf_col], errors='coerce'))
        pooled_intensity.append(pd.to_numeric(df['intensity'], errors='coerce'))
        pooled_frame.append(pd.to_numeric(df['frame'], errors='coerce'))

    # Concatenate pooled series and remove NaN / inf values
    pooled_uncertainty = pd.concat(pooled_uncertainty, ignore_index=True)
    pooled_uncertainty = pooled_uncertainty[np.isfinite(pooled_uncertainty)]

    pooled_psf = pd.concat(pooled_psf, ignore_index=True)
    pooled_psf = pooled_psf[np.isfinite(pooled_psf)]

    pooled_intensity = pd.concat(pooled_intensity, ignore_index=True)
    pooled_intensity = pooled_intensity[np.isfinite(pooled_intensity)]

    pooled_frame = pd.concat(pooled_frame, ignore_index=True)
    pooled_frame = pooled_frame[np.isfinite(pooled_frame)]

    # Calculate default percentile-based suggestions from pooled data
    loc_precision_max_default = pooled_uncertainty.quantile(0.95)
    psf_max_default = pooled_psf.quantile(0.95)
    intensity_min_default = pooled_intensity.quantile(0.05)
    intensity_max_default = pooled_intensity.quantile(0.95)
    frame_min_default = pooled_frame.quantile(0.05)

    print("\nDefault percentile values for filtering parameters (calculated from pooled data):")
    print(f"Localization precision max (95th percentile): {loc_precision_max_default}")
    print(f"PSF max (95th percentile): {psf_max_default}")
    print(f"Intensity min (5th percentile): {intensity_min_default}")
    print(f"Intensity max (95th percentile): {intensity_max_default}")
    print(f"Frame min (5th percentile): {frame_min_default}")

    # Let the user manually accept or modify the suggested thresholds
    thresholds = {
        'loc_precision_max': click.prompt(
            "Enter localization precision max",
            default=float(loc_precision_max_default),
            type=float
        ),
        'psf_max': click.prompt(
            "Enter PSF max",
            default=float(psf_max_default),
            type=float
        ),
        'intensity_min': click.prompt(
            "Enter intensity min",
            default=float(intensity_min_default),
            type=float
        ),
        'intensity_max': click.prompt(
            "Enter intensity max",
            default=float(intensity_max_default),
            type=float
        ),
        'frame_min': click.prompt(
            "Enter frame min",
            default=float(frame_min_default),
            type=float
        ),
    }

    return thresholds


# =============================================================================
# Function: filter_data
# =============================================================================
# Applies the selected filtering thresholds to one localization dataset.
#
# Parameters
# ----------
# locdata : LocData
#     Localization dataset to filter.
#
# file_type : str
#     Either "ELYRA" or "THUNDERSTORM".
#
# thresholds : dict
#     Dictionary containing the filtering thresholds:
#         - loc_precision_max
#         - psf_max
#         - intensity_min
#         - intensity_max
#         - frame_min
#
# Returns
# -------
# LocData
#     Filtered localization dataset wrapped in a LocData object.
#
# Notes
# -----
# A localization is kept only if it satisfies ALL conditions simultaneously.
# =============================================================================
def filter_data(locdata, file_type, thresholds):
    initial_count = len(locdata.data)

    # Determine which PSF column to use for this dataset
    psf_col = get_psf_column(locdata.data, file_type)

    # Verify that all required columns exist before filtering
    required_columns = ['uncertainty', 'intensity', 'frame', psf_col]
    for col in required_columns:
        if col not in locdata.data.columns:
            raise KeyError(f"The specified column '{col}' does not exist in the data.")

    # Apply all filters simultaneously
    filtered_data = locdata.data[
        (locdata.data['uncertainty'] <= thresholds['loc_precision_max']) &
        (locdata.data[psf_col] <= thresholds['psf_max']) &
        (locdata.data['intensity'] >= thresholds['intensity_min']) &
        (locdata.data['intensity'] <= thresholds['intensity_max']) &
        (locdata.data['frame'] >= thresholds['frame_min'])
    ]

    final_count = len(filtered_data)
    percentage_removed = 100 * (initial_count - final_count) / initial_count

    print(f"Initial localizations: {initial_count}")
    print(f"Final localizations: {final_count}")
    print(f"Percentage removed: {percentage_removed:.2f}%")

    return LocData.from_dataframe(filtered_data)


# =============================================================================
# Main workflow
# =============================================================================
# This block:
# 1. Loads files for two conditions
# 2. Pools all loaded data to suggest percentile-based filtering defaults
# 3. Prompts the user to confirm or edit those thresholds
# 4. Applies the same thresholds to all datasets
#
# Output structure
# ----------------
# all_filtered_locdata is a dictionary:
#
# {
#     "WT":   [(file_path_1, filtered_locdata_1), (file_path_2, filtered_locdata_2), ...],
#     "3NQ":  [(file_path_1, filtered_locdata_1), (file_path_2, filtered_locdata_2), ...]
# }
# =============================================================================
if __name__ == "__main__":
    import sys

    # Make matplotlib work nicely when the script is run inside Jupyter
    if 'ipykernel' in sys.modules:
        from IPython import get_ipython
        get_ipython().run_line_magic('matplotlib', 'inline')

    # -------------------------------------------------------------------------
    # Load data for condition 1
    # -------------------------------------------------------------------------
    condition1_name = "WT"
    file_type1, file_paths1 = select_files_for_condition(condition1_name)
    locdata_list1 = load_data(file_type1, file_paths1)

    # -------------------------------------------------------------------------
    # Load data for condition 2
    # -------------------------------------------------------------------------
    condition2_name = "3NQ"
    file_type2, file_paths2 = select_files_for_condition(condition2_name)
    locdata_list2 = load_data(file_type2, file_paths2)

    # Store condition name, file type, and loaded data together
    locdata_combined = [
        (condition1_name, file_type1, locdata_list1),
        (condition2_name, file_type2, locdata_list2)
    ]

    # Print the number of available CPU cores
    num_cpus = os.cpu_count()
    print(f"\nNumber of available CPU cores: {num_cpus}")

    # -------------------------------------------------------------------------
    # Build a pooled list of (file_type, locdata) across all conditions
    # This is used only for estimating default percentile thresholds.
    # -------------------------------------------------------------------------
    pooled_locdata_with_types = []

    for _, file_type, locdata_list in locdata_combined:
        for _, locdata in locdata_list:
            pooled_locdata_with_types.append((file_type, locdata))

    # Ask the user for filtering thresholds, using pooled percentiles as defaults
    print("\nEnter the filtering thresholds to be applied to both conditions:")
    thresholds = get_filtering_thresholds(pooled_locdata_with_types)

    # -------------------------------------------------------------------------
    # Filter all files condition by condition
    # -------------------------------------------------------------------------
    all_filtered_locdata = {}

    for condition_name, file_type, locdata_list in locdata_combined:
        print(f"\nProcessing condition: {condition_name}")
        filtered_locdata_list = []

        for file_name, locdata in tqdm(locdata_list, desc=f"Filtering data for {condition_name}"):
            filtered_locdata = filter_data(locdata, file_type, thresholds)
            filtered_locdata_list.append((file_name, filtered_locdata))

        all_filtered_locdata[condition_name] = filtered_locdata_list

    print("\nFiltering complete for both conditions.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# =============================================================================
# STEP 2 — Manual visual QC of filtered localization heatmaps
# =============================================================================
# This step shows one filtered dataset at a time as a log-scaled,
# intensity-weighted 2D heatmap. After viewing each plot, the user is asked
# whether to keep that dataset for downstream analysis.
#
# Output
# ------
# approved_locdata : dict
#     Dictionary with the same structure as all_filtered_locdata, but containing
#     only datasets that were manually approved.
#
# qc_summary_df : pandas.DataFrame
#     Table summarizing the QC decision for each file.
# =============================================================================

# -----------------------------------------------------------------------------
# Settings
# -----------------------------------------------------------------------------
condition_order = ['WT', '3NQ']   # order in which conditions are inspected
cmap = 'hot'                      # colormap for heatmap display
bins = 200                        # histogram bin number for x/y heatmap


# -----------------------------------------------------------------------------
# Containers for approved data and QC summary
# -----------------------------------------------------------------------------
approved_locdata = {condition: [] for condition in condition_order}
qc_summary = []


# -----------------------------------------------------------------------------
# Loop through each condition and each filtered file one by one
# -----------------------------------------------------------------------------
for condition_name in condition_order:
    filtered_locdata_list = all_filtered_locdata.get(condition_name, [])

    print(f"\nInspecting condition: {condition_name}")
    print("-" * 60)

    for file_idx, (file_name, filtered_locdata) in enumerate(filtered_locdata_list, start=1):
        data = filtered_locdata.data

        # ---------------------------------------------------------------------
        # Safety check: skip empty datasets
        # ---------------------------------------------------------------------
        if data.empty:
            print(f"\nFile {file_idx}: {file_name}")
            print("This dataset is empty after filtering and will be marked as rejected.")

            qc_summary.append({
                "condition": condition_name,
                "file_name": file_name,
                "n_localizations": 0,
                "decision": "rejected_empty"
            })
            continue

        # Extract localization coordinates and intensity
        x = data['position_x']
        y = data['position_y']
        intensities = data['intensity']

        # ---------------------------------------------------------------------
        # Create a 2D histogram weighted by localization intensity
        # ---------------------------------------------------------------------
        heatmap, xedges, yedges = np.histogram2d(
            x,
            y,
            bins=bins,
            weights=intensities,
            density=False
        )

        # Log scaling improves dynamic range visibility
        log_heatmap = np.log1p(heatmap)

        # ---------------------------------------------------------------------
        # Plot the heatmap for visual inspection
        # ---------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(6, 6))

        im = ax.imshow(
            log_heatmap.T,
            extent=[x.min(), x.max(), y.min(), y.max()],
            origin='lower',
            cmap=cmap,
            aspect='equal'
        )

        ax.set_title(
            f"{condition_name} | File {file_idx}\n{file_name}",
            fontsize=11
        )
        ax.set_xlabel("Position X")
        ax.set_ylabel("Position Y")

        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Log(Summed Intensity)")

        plt.show()

        # ---------------------------------------------------------------------
        # Ask the user whether to keep this file
        # ---------------------------------------------------------------------
        while True:
            decision = input("Keep this file for downstream analysis? (y/n): ").strip().lower()

            if decision in ["y", "yes"]:
                approved_locdata[condition_name].append((file_name, filtered_locdata))
                qc_summary.append({
                    "condition": condition_name,
                    "file_name": file_name,
                    "n_localizations": len(data),
                    "decision": "approved"
                })
                print("Approved.\n")
                break

            elif decision in ["n", "no"]:
                qc_summary.append({
                    "condition": condition_name,
                    "file_name": file_name,
                    "n_localizations": len(data),
                    "decision": "rejected"
                })
                print("Rejected.\n")
                break

            else:
                print("Invalid input. Please enter 'y' or 'n'.")


# -----------------------------------------------------------------------------
# Convert QC summary to a dataframe for easy review
# -----------------------------------------------------------------------------
qc_summary_df = pd.DataFrame(qc_summary)

print("\nManual QC complete.")
print("\nQC summary:")
print(qc_summary_df)

print("\nApproved files per condition:")
for condition_name, files in approved_locdata.items():
    print(f"{condition_name}: {len(files)} approved file(s)")

In [ ]:
import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.stats import mannwhitneyu
import seaborn as sns


# =============================================================================
# STEP 3A — Density-based square ROI picker with non-overlapping ROIs 
# =============================================================================
# This chunk:
# 1. Creates a dated output directory on the Desktop
# 2. Uses only manually approved datasets from the previous QC step
# 3. Displays each dataset in a square field of view
# 4. Recommends candidate ROIs based on localization density in a tiled grid
# 5. Ensures that the same points cannot be used in more than one ROI
# 6. Stores approved ROIs for downstream Ripley's H analysis
# 7. Plots number of localizations per ROI comparing WT and 3NQ
#
# Output
# ------
# analysis_output_dir : Path
#     Desktop output folder for all downstream figures and tables
#
# selected_regions_by_condition : dict
#     Nested dictionary storing approved ROIs for each dataset
#
# roi_summary_df : pandas.DataFrame
#     Summary table of selected ROIs
# =============================================================================


# -----------------------------------------------------------------------------
# Create dated output directory on Desktop
# -----------------------------------------------------------------------------
date_tag = datetime.now().strftime("%Y%m%d")
analysis_output_dir = Path.home() / "Desktop" / f"{date_tag}_glyco_analysis"
analysis_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Analysis output directory:\n{analysis_output_dir}")


# -----------------------------------------------------------------------------
# Plot style settings (kept consistent with later plots)
# -----------------------------------------------------------------------------
WT_COLOR = "#4C78A8"
NQ_COLOR = "#F58518"
PALETTE = {"WT": WT_COLOR, "3NQ": NQ_COLOR}
CONDITION_ORDER = ["WT", "3NQ"]

sns.set_style("white")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 600
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


# -----------------------------------------------------------------------------
# Helper: convert approved_locdata to dataframe structure used downstream
# -----------------------------------------------------------------------------
def prepare_condition_dataframe_list(approved_locdata, condition_name):
    """
    Convert approved LocData objects into dataframes with standardized x/y names.
    """
    dataset_list = []

    for file_name, loc in approved_locdata.get(condition_name, []):
        df = loc.data.copy().rename(columns={"position_x": "x", "position_y": "y"})
        dataset_list.append((file_name, df))

    return dataset_list


# -----------------------------------------------------------------------------
# Helper: define square plotting / tiling bounds
# -----------------------------------------------------------------------------
def get_square_bounds(df):
    """
    Return square bounds that fully contain the dataset.
    """
    xmin, xmax = df["x"].min(), df["x"].max()
    ymin, ymax = df["y"].min(), df["y"].max()

    x_center = 0.5 * (xmin + xmax)
    y_center = 0.5 * (ymin + ymax)

    x_span = xmax - xmin
    y_span = ymax - ymin
    side = max(x_span, y_span)

    square_xmin = x_center - side / 2
    square_xmax = x_center + side / 2
    square_ymin = y_center - side / 2
    square_ymax = y_center + side / 2

    return {
        "xmin": square_xmin,
        "xmax": square_xmax,
        "ymin": square_ymin,
        "ymax": square_ymax,
        "side": side
    }


# -----------------------------------------------------------------------------
# Helper: assign candidate tiled ROIs
# -----------------------------------------------------------------------------
def generate_tiled_roi_candidates(df, region_size=3000, min_points=20):
    """
    Generate non-overlapping square ROI candidates on a tiled grid and score
    them by localization density.
    """
    bounds = get_square_bounds(df)

    x_starts = np.arange(bounds["xmin"], bounds["xmax"] - region_size + 1e-9, region_size)
    y_starts = np.arange(bounds["ymin"], bounds["ymax"] - region_size + 1e-9, region_size)

    candidates = []

    for x0 in x_starts:
        for y0 in y_starts:
            x1 = x0 + region_size
            y1 = y0 + region_size

            # Half-open interval avoids point duplication at ROI borders
            in_tile = (
                (df["x"] >= x0) & (df["x"] < x1) &
                (df["y"] >= y0) & (df["y"] < y1)
            )

            n_points = int(in_tile.sum())

            if n_points >= min_points:
                density = n_points / (region_size ** 2)
                candidates.append({
                    "xmin": x0,
                    "xmax": x1,
                    "ymin": y0,
                    "ymax": y1,
                    "n_points": n_points,
                    "density": density
                })

    if len(candidates) == 0:
        return pd.DataFrame(columns=["xmin", "xmax", "ymin", "ymax", "n_points", "density"])

    candidates_df = pd.DataFrame(candidates).sort_values(
        by=["density", "n_points"],
        ascending=False
    ).reset_index(drop=True)

    return candidates_df


# -----------------------------------------------------------------------------
# Helper: extract ROI points using half-open intervals
# -----------------------------------------------------------------------------
def extract_roi_points(df, region):
    """
    Extract points inside one ROI using half-open bounds.
    """
    return df[
        (df["x"] >= region["xmin"]) & (df["x"] < region["xmax"]) &
        (df["y"] >= region["ymin"]) & (df["y"] < region["ymax"])
    ].copy()


# -----------------------------------------------------------------------------
# Helper: plot dataset with candidate ROI
# -----------------------------------------------------------------------------
def plot_candidate_roi(df, region, condition_name, dataset_idx, file_name, roi_idx):
    """
    Plot the full dataset in square view with one candidate ROI overlay.
    """
    bounds = get_square_bounds(df)

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.scatter(df["x"], df["y"], s=1, alpha=0.35, color="black")

    rect = patches.Rectangle(
        (region["xmin"], region["ymin"]),
        region["xmax"] - region["xmin"],
        region["ymax"] - region["ymin"],
        linewidth=2,
        edgecolor="red",
        facecolor="none"
    )
    ax.add_patch(rect)

    ax.set_xlim(bounds["xmin"], bounds["xmax"])
    ax.set_ylim(bounds["ymin"], bounds["ymax"])
    ax.set_aspect("equal")

    ax.set_title(
        f"{condition_name} | Dataset {dataset_idx + 1} | ROI candidate {roi_idx + 1}\n"
        f"{Path(file_name).name}",
        fontsize=11
    )
    ax.set_xlabel("Position X")
    ax.set_ylabel("Position Y")

    return fig, ax


# -----------------------------------------------------------------------------
# Main ROI selection function
# -----------------------------------------------------------------------------
def select_density_ranked_rois_for_condition(
    dataset_list,
    condition_name,
    region_size=3000,
    n_rois=5,
    min_points=20
):
    """
    Select non-overlapping ROIs for each dataset using density-ranked tiled
    candidate ROIs.
    """
    selected_regions_for_condition = []

    for dataset_idx, (file_name, df) in enumerate(dataset_list):
        print(f"\nSelecting ROIs for {condition_name} dataset {dataset_idx + 1}")
        print(f"File: {file_name}")

        candidates_df = generate_tiled_roi_candidates(
            df=df,
            region_size=region_size,
            min_points=min_points
        )

        if candidates_df.empty:
            print("No valid ROI candidates found for this dataset.")
            selected_regions_for_condition.append({
                "file_name": file_name,
                "regions": []
            })
            continue

        selected_regions = []
        candidate_pointer = 0

        while len(selected_regions) < n_rois and candidate_pointer < len(candidates_df):
            region = candidates_df.iloc[candidate_pointer].to_dict()

            fig, ax = plot_candidate_roi(
                df=df,
                region=region,
                condition_name=condition_name,
                dataset_idx=dataset_idx,
                file_name=file_name,
                roi_idx=len(selected_regions)
            )
            plt.show()

            print(
                f"Candidate stats | n_points = {int(region['n_points'])}, "
                f"density = {region['density']:.4e}"
            )

            decision = input("Accept this ROI? (y = yes / n = no / q = stop this dataset): ").strip().lower()
            plt.close(fig)

            if decision in ["y", "yes"]:
                selected_regions.append(region)
                print(f"ROI {len(selected_regions)} accepted.")
            elif decision in ["n", "no"]:
                print("ROI rejected. Showing next candidate.")
            elif decision in ["q", "quit", "stop"]:
                print("Stopping ROI selection for this dataset.")
                break
            else:
                print("Invalid input. Showing next candidate anyway.")

            candidate_pointer += 1

        selected_regions_for_condition.append({
            "file_name": file_name,
            "regions": selected_regions
        })

    return selected_regions_for_condition


# -----------------------------------------------------------------------------
# Helper: convert p-values to significance stars
# -----------------------------------------------------------------------------
def p_to_stars(p):
    """
    Convert p-value to significance annotation.
    """
    if pd.isna(p):
        return "NaN"
    elif p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


# -----------------------------------------------------------------------------
# Helper: run Mann–Whitney U test between WT and 3NQ
# -----------------------------------------------------------------------------
def compare_conditions_mannwhitney(df, metric_col):
    """
    Run a two-sided Mann–Whitney U test comparing WT and 3NQ for one metric.
    """
    wt_vals = pd.to_numeric(
        df.loc[df["condition"] == "WT", metric_col],
        errors="coerce"
    ).dropna()

    nq_vals = pd.to_numeric(
        df.loc[df["condition"] == "3NQ", metric_col],
        errors="coerce"
    ).dropna()

    if len(wt_vals) == 0 or len(nq_vals) == 0:
        return np.nan, np.nan

    stat, p_value = mannwhitneyu(wt_vals, nq_vals, alternative="two-sided")
    return stat, p_value


# -----------------------------------------------------------------------------
# Helper: add significance stars to an axis
# -----------------------------------------------------------------------------
def add_significance_annotation(ax, df, metric_col, y_padding_fraction=0.08):
    """
    Add WT vs 3NQ significance annotation above the plotted distributions.
    """
    _, p_value = compare_conditions_mannwhitney(df, metric_col)

    vals = pd.to_numeric(df[metric_col], errors="coerce").dropna()
    if len(vals) == 0:
        return

    y_min = vals.min()
    y_max = vals.max()
    y_range = y_max - y_min if y_max > y_min else 1

    y_line = y_max + y_padding_fraction * y_range
    y_text = y_line + 0.03 * y_range

    ax.plot(
        [0, 0, 1, 1],
        [y_line, y_line + 0.02 * y_range, y_line + 0.02 * y_range, y_line],
        lw=1.2,
        c="black"
    )

    ax.text(
        0.5,
        y_text,
        p_to_stars(p_value),
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold"
    )

    current_bottom, current_top = ax.get_ylim()
    new_top = max(current_top, y_max + 0.18 * y_range)
    ax.set_ylim(current_bottom, new_top)


# -----------------------------------------------------------------------------
# Helper: make x tick labels bold
# -----------------------------------------------------------------------------
def style_condition_ticklabels(ax):
    """
    Make WT / 3NQ x-axis tick labels bold.
    """
    for tick_label in ax.get_xticklabels():
        tick_label.set_fontweight("bold")
        tick_label.set_fontsize(12)


# -----------------------------------------------------------------------------
# Helper: add median lines to violin plots
# -----------------------------------------------------------------------------
def add_violin_median_lines(ax, df, metric_col, line_width=2.2, half_width=0.16):
    """
    Add a horizontal median line inside each violin.
    """
    for i, condition in enumerate(CONDITION_ORDER):
        vals = pd.to_numeric(
            df.loc[df["condition"] == condition, metric_col],
            errors="coerce"
        ).dropna()

        if len(vals) == 0:
            continue

        median_val = np.median(vals)

        ax.plot(
            [i - half_width, i + half_width],
            [median_val, median_val],
            color="white",
            linewidth=line_width,
            solid_capstyle="round",
            zorder=5
        )

        ax.plot(
            [i - half_width, i + half_width],
            [median_val, median_val],
            color="black",
            linewidth=0.8,
            solid_capstyle="round",
            zorder=6
        )


# -----------------------------------------------------------------------------
# Prepare approved datasets
# -----------------------------------------------------------------------------
condition_order = ["WT", "3NQ"]

wt_dataset_list = prepare_condition_dataframe_list(approved_locdata, "WT")
nq_dataset_list = prepare_condition_dataframe_list(approved_locdata, "3NQ")


# -----------------------------------------------------------------------------
# Run ROI selection
# -----------------------------------------------------------------------------
selected_regions_wt = select_density_ranked_rois_for_condition(
    dataset_list=wt_dataset_list,
    condition_name="WT",
    region_size=3000,
    n_rois=5,
    min_points=20
)

selected_regions_nq = select_density_ranked_rois_for_condition(
    dataset_list=nq_dataset_list,
    condition_name="3NQ",
    region_size=3000,
    n_rois=5,
    min_points=20
)


# -----------------------------------------------------------------------------
# Store in a condition-keyed structure for downstream analysis
# -----------------------------------------------------------------------------
selected_regions_by_condition = {
    "WT": selected_regions_wt,
    "3NQ": selected_regions_nq
}


# -----------------------------------------------------------------------------
# Build ROI summary table
# -----------------------------------------------------------------------------
roi_summary_rows = []

for condition_name, dataset_regions in selected_regions_by_condition.items():
    for dataset_idx, dataset_entry in enumerate(dataset_regions):
        file_name = dataset_entry["file_name"]
        for roi_idx, region in enumerate(dataset_entry["regions"], start=1):
            roi_summary_rows.append({
                "condition": condition_name,
                "dataset_index": dataset_idx + 1,
                "file_name": file_name,
                "roi_index": roi_idx,
                "xmin": region["xmin"],
                "xmax": region["xmax"],
                "ymin": region["ymin"],
                "ymax": region["ymax"],
                "n_points": int(region["n_points"]),
                "density": region["density"]
            })

roi_summary_df = pd.DataFrame(roi_summary_rows)


# -----------------------------------------------------------------------------
# Add condition medians to ROI summary table
# -----------------------------------------------------------------------------
if not roi_summary_df.empty:
    roi_medians = (
        roi_summary_df.groupby("condition")["n_points"]
        .median()
        .to_dict()
    )
else:
    roi_medians = {"WT": np.nan, "3NQ": np.nan}

roi_summary_df["condition_median_n_points"] = roi_summary_df["condition"].map(roi_medians)


# -----------------------------------------------------------------------------
# Save ROI summary table
# -----------------------------------------------------------------------------
roi_summary_path = analysis_output_dir / "roi_selection_summary.csv"
roi_summary_df.to_csv(roi_summary_path, index=False)

print("\nROI selection complete.")
print(f"ROI summary saved to:\n{roi_summary_path}")


# -----------------------------------------------------------------------------
# Plot number of localizations per ROI comparing WT and 3NQ
# -----------------------------------------------------------------------------
if not roi_summary_df.empty:
    plot_df = roi_summary_df.copy()
    plot_df = plot_df[plot_df["condition"].isin(CONDITION_ORDER)].copy()
    plot_df["n_points"] = pd.to_numeric(plot_df["n_points"], errors="coerce")
    plot_df = plot_df.dropna(subset=["n_points"])

    fig, ax = plt.subplots(figsize=(5.5, 6))

    sns.violinplot(
        data=plot_df,
        x="condition",
        y="n_points",
        hue="condition",
        order=CONDITION_ORDER,
        hue_order=CONDITION_ORDER,
        palette=PALETTE,
        inner=None,
        cut=0,
        linewidth=1.2,
        dodge=False,
        legend=False,
        ax=ax
    )

    add_violin_median_lines(ax, plot_df, "n_points")

    sns.swarmplot(
        data=plot_df,
        x="condition",
        y="n_points",
        order=CONDITION_ORDER,
        color="black",
        size=3,
        alpha=0.8,
        ax=ax
    )

    ax.set_title("Localizations per ROI", fontsize=16)
    ax.set_xlabel("")
    ax.set_ylabel("Number of localizations per ROI", fontsize=14)
    ax.tick_params(axis="x", labelsize=12)
    ax.tick_params(axis="y", labelsize=12)

    style_condition_ticklabels(ax)
    add_significance_annotation(ax, plot_df, "n_points")

    roi_count_plot_path = analysis_output_dir / "roi_localizations_per_roi_WT_vs_3NQ.png"
    fig.savefig(roi_count_plot_path, dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"ROI localization comparison figure saved to:\n{roi_count_plot_path}")

    print("\nMedian number of localizations per ROI:")
    for cond in CONDITION_ORDER:
        print(f"{cond}: {roi_medians.get(cond, np.nan)}")
else:
    print("\nNo approved ROIs were selected, so no ROI localization comparison plot was generated.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import distance
from scipy import stats


# =============================================================================
# STEP 3B — Ripley's H analysis on selected ROIs
# =============================================================================
# This chunk:
# 1. Uses the selected ROIs from the previous chunk
# 2. Calculates Ripley's H(r) for each ROI
# 3. Creates separate publication-style plots for WT and 3NQ
# 4. Generates both:
#       - mean ± SEM plots
#       - median + IQR plots
# 5. Marks the peak H(r) with a dot
# 6. Reports peak H and peak r in the figure legend only
# 7. Saves all plots as high-resolution PNG files
# =============================================================================


# -----------------------------------------------------------------------------
# Publication-style plotting settings
# -----------------------------------------------------------------------------
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 600
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

WT_COLOR = "#4C78A8"
NQ_COLOR = "#F58518"
INDIVIDUAL_CURVE_COLOR = "0.75"


# -----------------------------------------------------------------------------
# Ripley's H calculation
# -----------------------------------------------------------------------------
def manual_ripley_k(df_roi, max_radius, num_radii=50):
    """
    Calculate Ripley's K and H for one ROI.

    Parameters
    ----------
    df_roi : pandas.DataFrame
        ROI dataframe containing 'x' and 'y'.

    max_radius : float
        Maximum radius for Ripley's calculation.

    num_radii : int
        Number of radius values to evaluate.

    Returns
    -------
    radii : np.ndarray
    K_values : np.ndarray
    H_values : np.ndarray
    """
    points = df_roi[["x", "y"]].values
    num_points = len(points)

    if num_points < 2:
        radii = np.linspace(0, max_radius, num_radii)
        return radii, np.full_like(radii, np.nan), np.full_like(radii, np.nan)

    area = (df_roi["x"].max() - df_roi["x"].min()) * (df_roi["y"].max() - df_roi["y"].min())
    radii = np.linspace(0, max_radius, num_radii)
    K_values = np.zeros(num_radii)

    pairwise_distances = distance.cdist(points, points)

    for i, radius in enumerate(radii):
        within_radius = np.sum(pairwise_distances < radius) - num_points
        K_values[i] = (within_radius / num_points) * (area / num_points)

    H_values = np.sqrt(np.maximum(K_values / np.pi, 0)) - radii
    return radii, K_values, H_values


# -----------------------------------------------------------------------------
# Helper: collect ROI H-curves per condition
# -----------------------------------------------------------------------------
def collect_ripley_h_from_selected_rois(dataset_list, selected_regions_for_condition, max_radius=500, num_radii=50):
    """
    Calculate Ripley's H for all approved ROIs in one condition.

    Returns
    -------
    radii : np.ndarray
    h_curves : list of np.ndarray
    metadata_rows : list of dict
    """
    h_curves = []
    metadata_rows = []
    radii = np.linspace(0, max_radius, num_radii)

    dataset_lookup = {file_name: df for file_name, df in dataset_list}

    for dataset_idx, dataset_entry in enumerate(selected_regions_for_condition, start=1):
        file_name = dataset_entry["file_name"]
        regions = dataset_entry["regions"]
        df = dataset_lookup[file_name]

        for roi_idx, region in enumerate(regions, start=1):
            df_roi = extract_roi_points(df, region)

            if len(df_roi) < 2:
                continue

            radii, _, h_values = manual_ripley_k(
                df_roi=df_roi,
                max_radius=max_radius,
                num_radii=num_radii
            )

            if np.all(np.isnan(h_values)):
                continue

            h_curves.append(h_values)

            metadata_rows.append({
                "dataset_index": dataset_idx,
                "file_name": file_name,
                "roi_index": roi_idx,
                "n_points": len(df_roi)
            })

    return radii, h_curves, metadata_rows


# -----------------------------------------------------------------------------
# Helper: peak calculation
# -----------------------------------------------------------------------------
def get_curve_peak(radii, h_values):
    """
    Return the radius and value of the maximum H(r).
    """
    peak_idx = int(np.nanargmax(h_values))
    return radii[peak_idx], h_values[peak_idx]


# -----------------------------------------------------------------------------
# Helper: plot one condition using mean ± SEM
# -----------------------------------------------------------------------------
def plot_ripley_mean_sem_single_condition(
    radii,
    h_curve_list,
    condition_name,
    color,
    save_dir
):
    """
    Plot a single-condition Ripley's H figure using mean ± SEM.
    Individual ROI curves are shown in gray.
    """
    if len(h_curve_list) == 0:
        print(f"No valid Ripley H curves found for {condition_name}. Skipping mean ± SEM plot.")
        return

    h_array = np.vstack(h_curve_list)
    mean_h = np.nanmean(h_array, axis=0)
    sem_h = stats.sem(h_array, axis=0, nan_policy="omit")

    peak_r, peak_h = get_curve_peak(radii, mean_h)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))

    # Individual ROI curves
    for h_curve in h_curve_list:
        ax.plot(radii, h_curve, color=INDIVIDUAL_CURVE_COLOR, linewidth=1.0, alpha=0.8)

    # Mean ± SEM
    ax.plot(
        radii,
        mean_h,
        color=color,
        linewidth=2.5,
        label=f"Mean H(r) | Peak H = {peak_h:.2f}, r = {peak_r:.1f}"
    )
    ax.fill_between(
        radii,
        mean_h - sem_h,
        mean_h + sem_h,
        color=color,
        alpha=0.20,
        label="SEM"
    )

    # CSR reference
    ax.axhline(0, color="black", linestyle="--", linewidth=1.2)

    # Peak dot only
    ax.scatter([peak_r], [peak_h], s=40, color=color, zorder=5)

    ax.set_title(f"Ripley's H(r) — {condition_name} (Mean ± SEM)")
    ax.set_xlabel("Radius r")
    ax.set_ylabel("Ripley's H(r)")
    ax.legend(frameon=False, loc="best")

    save_path = Path(save_dir) / f"ripley_H_{condition_name}_mean_SEM.png"
    fig.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"Saved:\n{save_path}")


# -----------------------------------------------------------------------------
# Helper: plot one condition using median + IQR
# -----------------------------------------------------------------------------
def plot_ripley_median_iqr_single_condition(
    radii,
    h_curve_list,
    condition_name,
    color,
    save_dir
):
    """
    Plot a single-condition Ripley's H figure using median + IQR.
    Individual ROI curves are shown in gray.
    """
    if len(h_curve_list) == 0:
        print(f"No valid Ripley H curves found for {condition_name}. Skipping median + IQR plot.")
        return

    h_array = np.vstack(h_curve_list)
    median_h = np.nanmedian(h_array, axis=0)
    q25 = np.nanpercentile(h_array, 25, axis=0)
    q75 = np.nanpercentile(h_array, 75, axis=0)

    peak_r, peak_h = get_curve_peak(radii, median_h)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))

    # Individual ROI curves
    for h_curve in h_curve_list:
        ax.plot(radii, h_curve, color=INDIVIDUAL_CURVE_COLOR, linewidth=1.0, alpha=0.8)

    # Median + IQR
    ax.plot(
        radii,
        median_h,
        color=color,
        linewidth=2.5,
        label=f"Median H(r) | Peak H = {peak_h:.2f}, r = {peak_r:.1f}"
    )
    ax.fill_between(
        radii,
        q25,
        q75,
        color=color,
        alpha=0.20,
        label="IQR"
    )

    # CSR reference
    ax.axhline(0, color="black", linestyle="--", linewidth=1.2, label="CSR: H(r) = 0")

    # Peak dot only
    ax.scatter([peak_r], [peak_h], s=40, color=color, zorder=5)

    ax.set_title(f"Ripley's H(r) — {condition_name} (Median + IQR)")
    ax.set_xlabel("Radius r")
    ax.set_ylabel("Ripley's H(r)")
    ax.legend(frameon=False, loc="best")

    save_path = Path(save_dir) / f"ripley_H_{condition_name}_median_IQR.png"
    fig.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"Saved:\n{save_path}")


# -----------------------------------------------------------------------------
# Prepare approved datasets again
# -----------------------------------------------------------------------------
wt_dataset_list = prepare_condition_dataframe_list(approved_locdata, "WT")
nq_dataset_list = prepare_condition_dataframe_list(approved_locdata, "3NQ")


# -----------------------------------------------------------------------------
# Calculate Ripley's H for selected ROIs
# -----------------------------------------------------------------------------
max_radius = 500
num_radii = 50

radii_wt, wt_h_curves, wt_meta = collect_ripley_h_from_selected_rois(
    dataset_list=wt_dataset_list,
    selected_regions_for_condition=selected_regions_by_condition["WT"],
    max_radius=max_radius,
    num_radii=num_radii
)

radii_nq, nq_h_curves, nq_meta = collect_ripley_h_from_selected_rois(
    dataset_list=nq_dataset_list,
    selected_regions_for_condition=selected_regions_by_condition["3NQ"],
    max_radius=max_radius,
    num_radii=num_radii
)

# Shared radii axis
radii = radii_wt


# -----------------------------------------------------------------------------
# Save ROI-level metadata for traceability
# -----------------------------------------------------------------------------
ripley_roi_metadata = pd.DataFrame(
    [{"condition": "WT", **row} for row in wt_meta] +
    [{"condition": "3NQ", **row} for row in nq_meta]
)

ripley_metadata_path = analysis_output_dir / "ripley_roi_metadata.csv"
ripley_roi_metadata.to_csv(ripley_metadata_path, index=False)

print(f"Ripley ROI metadata saved to:\n{ripley_metadata_path}")


# -----------------------------------------------------------------------------
# Generate and save separate Ripley plots
# -----------------------------------------------------------------------------
plot_ripley_mean_sem_single_condition(
    radii=radii,
    h_curve_list=wt_h_curves,
    condition_name="WT",
    color=WT_COLOR,
    save_dir=analysis_output_dir
)

plot_ripley_mean_sem_single_condition(
    radii=radii,
    h_curve_list=nq_h_curves,
    condition_name="3NQ",
    color=NQ_COLOR,
    save_dir=analysis_output_dir
)

plot_ripley_median_iqr_single_condition(
    radii=radii,
    h_curve_list=wt_h_curves,
    condition_name="WT",
    color=WT_COLOR,
    save_dir=analysis_output_dir
)

plot_ripley_median_iqr_single_condition(
    radii=radii,
    h_curve_list=nq_h_curves,
    condition_name="3NQ",
    color=NQ_COLOR,
    save_dir=analysis_output_dir
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import distance
from sklearn.cluster import DBSCAN
from IPython.display import display


# =============================================================================
# STEP 4 — DBSCAN clustering on selected ROIs
# =============================================================================
# This chunk:
# 1. Prompts the user for DBSCAN parameters (eps and min_samples)
# 2. Applies DBSCAN to each selected ROI in each approved dataset
# 3. Calculates clustering summary metrics for each ROI
# 4. Saves the summary automatically to analysis_output_dir
# 5. Keeps both the summary table and ROI-level clustering results in memory
#
# Outputs
# -------
# dbscan_summary_df : pandas.DataFrame
#     Summary table of clustering metrics for each ROI
#
# dbscan_results_by_condition : dict
#     Nested dictionary storing clustered ROI-level dataframes and metadata
# =============================================================================


# -----------------------------------------------------------------------------
# Helper: calculate cluster metrics from one clustered ROI dataframe
# -----------------------------------------------------------------------------
def calculate_dbscan_cluster_metrics(clustered_df):
    """
    Calculate summary clustering metrics from a dataframe containing DBSCAN
    labels for one ROI.

    Parameters
    ----------
    clustered_df : pandas.DataFrame
        ROI dataframe with columns 'x', 'y', and 'labels'.

    Returns
    -------
    metrics : dict
        Dictionary containing:
            - n_clusters
            - fraction_in_cluster
            - localization_density
            - mean_cluster_diameter
    """
    labels = clustered_df["labels"]
    unique_labels = set(labels)

    # Number of non-noise clusters
    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)

    # Fraction of all localizations assigned to a cluster
    clustered_points = clustered_df[clustered_df["labels"] != -1]
    fraction_in_cluster = len(clustered_points) / len(clustered_df) if len(clustered_df) > 0 else np.nan

    # Localization density in localizations / µm²
    area_nm2 = (clustered_df["x"].max() - clustered_df["x"].min()) * (clustered_df["y"].max() - clustered_df["y"].min())
    area_um2 = area_nm2 / 1e6
    localization_density = len(clustered_df) / area_um2 if area_um2 > 0 else np.nan

    # Mean cluster diameter (maximum pairwise distance within each cluster)
    cluster_diameters = []
    for label in unique_labels:
        if label == -1:
            continue

        cluster_points = clustered_df.loc[clustered_df["labels"] == label, ["x", "y"]].values

        if len(cluster_points) > 1:
            max_distance = distance.pdist(cluster_points).max()
        else:
            max_distance = 0.0

        cluster_diameters.append(max_distance)

    mean_cluster_diameter = np.mean(cluster_diameters) if cluster_diameters else 0.0

    return {
        "n_clusters": n_clusters,
        "fraction_in_cluster": fraction_in_cluster,
        "localization_density": localization_density,
        "mean_cluster_diameter": mean_cluster_diameter
    }


# -----------------------------------------------------------------------------
# Helper: run DBSCAN on a single ROI
# -----------------------------------------------------------------------------
def run_dbscan_on_roi(df_roi, eps, min_samples):
    """
    Apply DBSCAN clustering to one ROI dataframe.

    Parameters
    ----------
    df_roi : pandas.DataFrame
        ROI dataframe with 'x' and 'y' columns.

    eps : float
        DBSCAN eps parameter in the same units as x/y coordinates (typically nm).

    min_samples : int
        DBSCAN min_samples parameter.

    Returns
    -------
    clustered_df : pandas.DataFrame
        Copy of ROI dataframe with a new 'labels' column.
    """
    clustered_df = df_roi.copy()

    if clustered_df.empty:
        clustered_df["labels"] = pd.Series(dtype=int)
        return clustered_df

    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    clustered_df.loc[:, "labels"] = dbscan.fit_predict(clustered_df[["x", "y"]])

    return clustered_df


# -----------------------------------------------------------------------------
# Main DBSCAN function
# -----------------------------------------------------------------------------
def perform_dbscan_clustering_on_selected_rois(
    wt_dataset_list,
    nq_dataset_list,
    selected_regions_by_condition,
    eps,
    min_samples,
    save_dir
):
    """
    Perform DBSCAN clustering on all selected ROIs from WT and 3NQ conditions.

    Parameters
    ----------
    wt_dataset_list : list of tuples
        List of (file_name, dataframe) for WT.

    nq_dataset_list : list of tuples
        List of (file_name, dataframe) for 3NQ.

    selected_regions_by_condition : dict
        Dictionary with keys 'WT' and '3NQ', where each value is a list of
        dataset entries with selected ROIs.

    eps : float
        DBSCAN eps parameter.

    min_samples : int
        DBSCAN min_samples parameter.

    save_dir : str or Path
        Directory where the summary CSV should be saved.

    Returns
    -------
    summary_df : pandas.DataFrame
        ROI-level clustering summary.

    dbscan_results_by_condition : dict
        Nested dictionary storing clustered ROI dataframes and metadata.
    """
    results_rows = []

    dbscan_results_by_condition = {
        "WT": [],
        "3NQ": []
    }

    dataset_lookup = {
        "WT": {file_name: df for file_name, df in wt_dataset_list},
        "3NQ": {file_name: df for file_name, df in nq_dataset_list}
    }

    for condition_name in ["WT", "3NQ"]:
        dataset_entries = selected_regions_by_condition[condition_name]

        for dataset_idx, dataset_entry in enumerate(dataset_entries, start=1):
            file_name = dataset_entry["file_name"]
            regions = dataset_entry["regions"]
            df = dataset_lookup[condition_name][file_name]

            roi_results = []

            for roi_idx, region in enumerate(regions, start=1):
                df_roi = extract_roi_points(df, region)

                if df_roi.empty:
                    continue

                clustered_df = run_dbscan_on_roi(
                    df_roi=df_roi,
                    eps=eps,
                    min_samples=min_samples
                )

                metrics = calculate_dbscan_cluster_metrics(clustered_df)

                # Store summary row for later statistics/plotting
                results_rows.append({
                    "condition": condition_name,
                    "dataset_index": dataset_idx,
                    "file_name": file_name,
                    "roi_index": roi_idx,
                    "eps_nm": eps,
                    "min_samples": min_samples,
                    "n_localizations": len(clustered_df),
                    "n_clusters": metrics["n_clusters"],
                    "fraction_in_cluster": metrics["fraction_in_cluster"],
                    "localization_density": metrics["localization_density"],
                    "mean_cluster_diameter_nm": metrics["mean_cluster_diameter"]
                })

                # Store full clustered ROI data for later visualization if needed
                roi_results.append({
                    "roi_index": roi_idx,
                    "region": region,
                    "clustered_df": clustered_df,
                    "metrics": metrics
                })

            dbscan_results_by_condition[condition_name].append({
                "dataset_index": dataset_idx,
                "file_name": file_name,
                "roi_results": roi_results
            })

    # Build summary dataframe
    summary_df = pd.DataFrame(results_rows)

    print("\nDBSCAN clustering results summary:")
    display(summary_df)

    # Save summary automatically into the analysis folder
    save_name = f"dbscan_summary_eps_{eps:g}_minsamples_{min_samples}.csv"
    save_path = analysis_output_dir / save_name
    summary_df.to_csv(save_path, index=False)

    print(f"\nDBSCAN summary saved to:\n{save_path}")

    return summary_df, dbscan_results_by_condition


# -----------------------------------------------------------------------------
# Prompt user for DBSCAN parameters
# -----------------------------------------------------------------------------
eps = float(input("Enter DBSCAN eps (in nm): ").strip())
min_samples = int(input("Enter DBSCAN min_samples: ").strip())


# -----------------------------------------------------------------------------
# Run DBSCAN on the selected ROIs
# -----------------------------------------------------------------------------
dbscan_summary_df, dbscan_results_by_condition = perform_dbscan_clustering_on_selected_rois(
    wt_dataset_list=wt_dataset_list,
    nq_dataset_list=nq_dataset_list,
    selected_regions_by_condition=selected_regions_by_condition,
    eps=eps,
    min_samples=min_samples,
    save_dir=analysis_output_dir
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import seaborn as sns

# =============================================================================
# STEP 5 — Plot DBSCAN summary metrics for WT vs 3NQ as separate figures
# =============================================================================
# This chunk creates 4 separate comparison figures from the
# ROI-level DBSCAN summary table.
#
# Plots included
# --------------
# 1. Number of clusters per ROI           -> violin + swarm + median line
# 2. Fraction of localizations in cluster -> box + swarm
# 3. Mean cluster diameter (nm)           -> violin + swarm + median line
# 4. Localization density (locs / µm²)    -> box + swarm
#
# Statistics
# ----------
# For each metric, WT and 3NQ are compared using a two-sided
# Mann–Whitney U test.
# The plot shows significance as asterisks, while exact p-values are saved to CSV.
#
# Additional outputs
# ------------------
# Saves two CSV files:
# 1. Summary statistics by condition for each metric
# 2. WT vs 3NQ comparison statistics for each metric
#
# Input required
# --------------
# dbscan_summary_df
#     DataFrame generated in the previous DBSCAN chunk.
#
# Output
# ------
# Saves 4 separate PNG figures to analysis_output_dir.
# Saves 2 CSV summary tables to analysis_output_dir.
# =============================================================================


# -----------------------------------------------------------------------------
# Plot style settings
# -----------------------------------------------------------------------------
WT_COLOR = "#4C78A8"
NQ_COLOR = "#F58518"
PALETTE = {"WT": WT_COLOR, "3NQ": NQ_COLOR}
CONDITION_ORDER = ["WT", "3NQ"]

sns.set_style("white")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 600
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


# -----------------------------------------------------------------------------
# Safety check: make sure the expected columns exist
# -----------------------------------------------------------------------------
required_columns = [
    "condition",
    "n_clusters",
    "fraction_in_cluster",
    "mean_cluster_diameter_nm",
    "localization_density"
]

missing_columns = [col for col in required_columns if col not in dbscan_summary_df.columns]
if missing_columns:
    raise KeyError(f"The following required columns are missing from dbscan_summary_df: {missing_columns}")


# -----------------------------------------------------------------------------
# Helper: convert p-values to significance stars
# -----------------------------------------------------------------------------
def p_to_stars(p):
    """
    Convert p-value to significance annotation.
    """
    if pd.isna(p):
        return "NaN"
    elif p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


# -----------------------------------------------------------------------------
# Helper: run Mann–Whitney U test between WT and 3NQ
# -----------------------------------------------------------------------------
def compare_conditions_mannwhitney(df, metric_col):
    """
    Run a two-sided Mann–Whitney U test comparing WT and 3NQ for one metric.
    """
    wt_vals = pd.to_numeric(
        df.loc[df["condition"] == "WT", metric_col],
        errors="coerce"
    ).dropna()

    nq_vals = pd.to_numeric(
        df.loc[df["condition"] == "3NQ", metric_col],
        errors="coerce"
    ).dropna()

    if len(wt_vals) == 0 or len(nq_vals) == 0:
        return np.nan, np.nan

    stat, p_value = mannwhitneyu(wt_vals, nq_vals, alternative="two-sided")
    return stat, p_value


# -----------------------------------------------------------------------------
# Helper: add significance annotation to an axis
# -----------------------------------------------------------------------------
def add_significance_annotation(ax, df, metric_col, y_padding_fraction=0.08):
    """
    Add WT vs 3NQ significance annotation above the plotted distributions.
    Displays significance as stars rather than exact p-values.
    """
    _, p_value = compare_conditions_mannwhitney(df, metric_col)

    vals = pd.to_numeric(df[metric_col], errors="coerce").dropna()
    if len(vals) == 0:
        return

    y_min = vals.min()
    y_max = vals.max()
    y_range = y_max - y_min if y_max > y_min else 1

    y_line = y_max + y_padding_fraction * y_range
    y_text = y_line + 0.03 * y_range

    ax.plot(
        [0, 0, 1, 1],
        [y_line, y_line + 0.02 * y_range, y_line + 0.02 * y_range, y_line],
        lw=1.2,
        c="black"
    )

    ax.text(
        0.5,
        y_text,
        p_to_stars(p_value),
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold"
    )

    current_bottom, current_top = ax.get_ylim()
    new_top = max(current_top, y_max + 0.18 * y_range)
    ax.set_ylim(current_bottom, new_top)


# -----------------------------------------------------------------------------
# Helper: make x tick labels bold
# -----------------------------------------------------------------------------
def style_condition_ticklabels(ax):
    """
    Make WT / 3NQ x-axis tick labels bold.
    """
    for tick_label in ax.get_xticklabels():
        tick_label.set_fontweight("bold")
        tick_label.set_fontsize(12)


# -----------------------------------------------------------------------------
# Helper: add median lines to violin plots
# -----------------------------------------------------------------------------
def add_violin_median_lines(ax, df, metric_col, line_width=2.2, half_width=0.16):
    """
    Add a horizontal median line inside each violin.
    """
    for i, condition in enumerate(CONDITION_ORDER):
        vals = pd.to_numeric(
            df.loc[df["condition"] == condition, metric_col],
            errors="coerce"
        ).dropna()

        if len(vals) == 0:
            continue

        median_val = np.median(vals)

        ax.plot(
            [i - half_width, i + half_width],
            [median_val, median_val],
            color="white",
            linewidth=line_width,
            solid_capstyle="round",
            zorder=5
        )

        ax.plot(
            [i - half_width, i + half_width],
            [median_val, median_val],
            color="black",
            linewidth=0.8,
            solid_capstyle="round",
            zorder=6
        )


# -----------------------------------------------------------------------------
# Helper: compute summary statistics for one metric and one condition
# -----------------------------------------------------------------------------
def compute_metric_summary_stats(df, metric_col, condition_name):
    """
    Compute descriptive statistics for one metric in one condition.
    """
    vals = pd.to_numeric(
        df.loc[df["condition"] == condition_name, metric_col],
        errors="coerce"
    ).dropna()

    if len(vals) == 0:
        return {
            "condition": condition_name,
            "metric": metric_col,
            "n": 0,
            "mean": np.nan,
            "median": np.nan,
            "std": np.nan,
            "sem": np.nan,
            "min": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "iqr": np.nan,
            "max": np.nan
        }

    q25 = np.percentile(vals, 25)
    q75 = np.percentile(vals, 75)

    return {
        "condition": condition_name,
        "metric": metric_col,
        "n": int(len(vals)),
        "mean": float(np.mean(vals)),
        "median": float(np.median(vals)),
        "std": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
        "sem": float(np.std(vals, ddof=1) / np.sqrt(len(vals))) if len(vals) > 1 else 0.0,
        "min": float(np.min(vals)),
        "q25": float(q25),
        "q75": float(q75),
        "iqr": float(q75 - q25),
        "max": float(np.max(vals))
    }


# -----------------------------------------------------------------------------
# Plot specifications
# -----------------------------------------------------------------------------
plot_specs = [
    {
        "metric": "n_clusters",
        "title": "Number of Clusters per ROI",
        "ylabel": "Number of clusters",
        "plot_type": "violin",
        "filename": f"dbscan_n_clusters_WT_vs_3NQ_eps_{eps:g}_minsamples_{min_samples}.png"
    },
    {
        "metric": "fraction_in_cluster",
        "title": "Fraction of Localizations in Clusters",
        "ylabel": "Fraction in cluster",
        "plot_type": "box",
        "filename": f"dbscan_fraction_in_cluster_WT_vs_3NQ_eps_{eps:g}_minsamples_{min_samples}.png"
    },
    {
        "metric": "mean_cluster_diameter_nm",
        "title": "Mean Cluster Diameter",
        "ylabel": "Mean cluster diameter (nm)",
        "plot_type": "violin",
        "filename": f"dbscan_mean_cluster_diameter_WT_vs_3NQ_eps_{eps:g}_minsamples_{min_samples}.png"
    },
    {
        "metric": "localization_density",
        "title": "Localization Density",
        "ylabel": "Localization density (locs / µm²)",
        "plot_type": "box",
        "filename": f"dbscan_localization_density_WT_vs_3NQ_eps_{eps:g}_minsamples_{min_samples}.png"
    }
]


# -----------------------------------------------------------------------------
# Build and save summary statistics tables
# -----------------------------------------------------------------------------
summary_stats_rows = []
comparison_rows = []

for spec in plot_specs:
    metric = spec["metric"]

    for condition_name in CONDITION_ORDER:
        summary_stats_rows.append(
            compute_metric_summary_stats(dbscan_summary_df, metric, condition_name)
        )

    wt_vals = pd.to_numeric(
        dbscan_summary_df.loc[dbscan_summary_df["condition"] == "WT", metric],
        errors="coerce"
    ).dropna()

    nq_vals = pd.to_numeric(
        dbscan_summary_df.loc[dbscan_summary_df["condition"] == "3NQ", metric],
        errors="coerce"
    ).dropna()

    stat, p_value = compare_conditions_mannwhitney(dbscan_summary_df, metric)

    comparison_rows.append({
        "metric": metric,
        "wt_n": int(len(wt_vals)),
        "nq_n": int(len(nq_vals)),
        "mannwhitney_u_statistic": stat,
        "p_value": p_value,
        "significance": p_to_stars(p_value)
    })

dbscan_summary_stats_df = pd.DataFrame(summary_stats_rows)
dbscan_comparison_stats_df = pd.DataFrame(comparison_rows)

summary_stats_path = analysis_output_dir / f"dbscan_summary_stats_by_condition_eps_{eps:g}_minsamples_{min_samples}.csv"
comparison_stats_path = analysis_output_dir / f"dbscan_condition_comparisons_eps_{eps:g}_minsamples_{min_samples}.csv"

dbscan_summary_stats_df.to_csv(summary_stats_path, index=False)
dbscan_comparison_stats_df.to_csv(comparison_stats_path, index=False)

print(f"Summary statistics saved to:\n{summary_stats_path}")
print(f"Condition comparison statistics saved to:\n{comparison_stats_path}")


# -----------------------------------------------------------------------------
# Create each plot separately
# -----------------------------------------------------------------------------
for spec in plot_specs:
    metric = spec["metric"]
    title = spec["title"]
    ylabel = spec["ylabel"]
    plot_type = spec["plot_type"]
    filename = spec["filename"]

    plot_df = dbscan_summary_df.copy()
    plot_df = plot_df[plot_df["condition"].isin(CONDITION_ORDER)].copy()
    plot_df[metric] = pd.to_numeric(plot_df[metric], errors="coerce")
    plot_df = plot_df.dropna(subset=[metric])

    fig, ax = plt.subplots(figsize=(5.5, 6))

    if plot_type == "violin":
        sns.violinplot(
            data=plot_df,
            x="condition",
            y=metric,
            hue="condition",
            order=CONDITION_ORDER,
            hue_order=CONDITION_ORDER,
            palette=PALETTE,
            inner=None,
            cut=0,
            linewidth=1.2,
            dodge=False,
            legend=False,
            ax=ax
        )

        add_violin_median_lines(ax, plot_df, metric)

        sns.swarmplot(
            data=plot_df,
            x="condition",
            y=metric,
            order=CONDITION_ORDER,
            color="black",
            size=3,
            alpha=0.8,
            ax=ax
        )

    elif plot_type == "box":
        sns.boxplot(
            data=plot_df,
            x="condition",
            y=metric,
            hue="condition",
            order=CONDITION_ORDER,
            hue_order=CONDITION_ORDER,
            palette=PALETTE,
            width=0.5,
            fliersize=0,
            linewidth=1.2,
            dodge=False,
            legend=False,
            ax=ax
        )

        sns.swarmplot(
            data=plot_df,
            x="condition",
            y=metric,
            order=CONDITION_ORDER,
            color="black",
            size=3,
            alpha=0.8,
            ax=ax
        )

    ax.set_title(title, fontsize=16)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=14)
    ax.tick_params(axis="x", labelsize=12)
    ax.tick_params(axis="y", labelsize=12)

    style_condition_ticklabels(ax)

    if metric == "fraction_in_cluster":
        vals = plot_df[metric].dropna()
        if len(vals) > 0:
            min_val = vals.min()
            lower_limit = np.floor(min_val * 20) / 20
            upper_limit = 1.0
            ax.set_ylim(lower_limit, upper_limit)

    add_significance_annotation(ax, plot_df, metric)

    plt.show()

    save_path = analysis_output_dir / filename
    fig.savefig(save_path, dpi=600, bbox_inches="tight")
    print(f"Figure saved to:\n{save_path}")

    plt.close(fig)